In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio, HTML
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Style configuration for dashboard
plt.style.use('seaborn-v0_8-darkgrid')

# --- 1. CONFIGURATION ---
BASE_PATH = Path("/data4/Henri/j3/framewiseSpeakerCounting")
DATASET_BASE = BASE_PATH / "databases/precomputed"
CHECKPOINT_BASE = BASE_PATH / "checkpoints"
PRED_BASE = BASE_PATH / "predictions"
RESULTS_BASE = BASE_PATH / "results"

# Default paths - can be modified if needed
DEFAULT_EXP_NAME = "J2_BXLS_deactivation_50ms" # "Klaus_PALD_3D_1" 
DATASET_PATH = DATASET_BASE / DEFAULT_EXP_NAME
PRED_PATH = PRED_BASE / f"{DEFAULT_EXP_NAME}_WGMSC_Feature_Extractor_GRU_estimator"

# --- 2. DATA LOADING ---
def load_data(scenario_id, split='test'):
    """Loads scenario, stft, feature, and prediction data."""
    data_dict = {}
    
    try:
        # Locate folders (assumes structure from Test_data.ipynb)
        stft_dir = list((DATASET_PATH / "stft").glob("*"))[0]
        feature_dir = list((DATASET_PATH / "features").glob("*"))[0]
        
        # Construct file paths
        scenario_file = DATASET_PATH / split / f"scenario_{scenario_id}.pt"
        stft_file = stft_dir / split / f"scenario_{scenario_id}.pt"
        feature_file = feature_dir / split / f"scenario_{scenario_id}.pt"
        
        # Prediction file pattern
        pred_pattern = f"*_{split}_generator_{scenario_id}.pt"
        pred_files = list(PRED_PATH.glob(pred_pattern))
        
        # Load Files
        if scenario_file.exists():
            data_dict["scenario_data"] = torch.load(scenario_file, weights_only=False)
        else:
            return None, f"Scenario file not found: {scenario_file}"
            
        if stft_file.exists():
            data_dict["stft_data"] = torch.load(stft_file, weights_only=False)
            
        if feature_file.exists():
            data_dict["feature_data"] = torch.load(feature_file, weights_only=False)
            
        if pred_files:
            data_dict["prediction_data"] = torch.load(pred_files[0], weights_only=False)
        else:
            print("Warning: Prediction file not found.")
            
        return data_dict, "Success"
        
    except Exception as e:
        return None, str(e)

# --- 3. HELPER: Pre-process & Time Axis ---
def preprocess_data(data_dict):
    """Extracts common variables and time axes."""
    if "scenario_data" not in data_dict: return None
    
    meta = data_dict["scenario_data"]["meta"]
    params = meta["scenario_params"]
    
    # Sampling Params
    fs = 16000
    hop = 256
    if "transform" in params:
        t = params["transform"]
        if hasattr(t, 'sampling_frequency'): fs = t.sampling_frequency
        if hasattr(t, 'hop_length'): hop = t.hop_length
        elif hasattr(t, 'hop_size'): hop = t.hop_size
            
    # Mixture / Components for Waveforms
    references = meta.get('references', {})
    sad_samples = meta.get('sad_samples', {})
    
    # Sort speakers by start time
    speaker_start_times = []
    for k in references.keys():
        if k == 'noise': continue
        # Find start
        start_idx = float('inf')
        if k in sad_samples:
            v = sad_samples[k]
            if isinstance(v, torch.Tensor): v = v.cpu().numpy()
            active = np.where(v > 0.5)[0]
            if len(active) > 0: start_idx = active[0]
        speaker_start_times.append((k, start_idx))
    speaker_start_times.sort(key=lambda x: x[1])
    sorted_speakers = [x[0] for x in speaker_start_times]
    
    ordered_labels = ['noisy']
    if 'noise' in references: ordered_labels.append('noise')
    ordered_labels.extend(sorted_speakers)
    
    # Create signals dict
    signals = {}
    vads = {}
    
    # Mixture
    if references:
        first_ref = next(iter(references.values()))
        mix = torch.zeros_like(first_ref)
        for k, v in references.items(): mix += v
        if mix.ndim == 2: mix = mix[0]
        signals['noisy'] = mix.float().cpu().numpy()
        vads['noisy'] = None
    
    # Components
    for k in ordered_labels:
        if k == 'noisy': continue
        sig = references[k]
        if sig.ndim == 2: sig = sig[0]
        signals[k] = sig.float().cpu().numpy()
        
        v = None
        if k in sad_samples:
            v = sad_samples[k]
            if isinstance(v, torch.Tensor): v = v.cpu().numpy()
        vads[k] = v
        
    # Time Axes
    # Waveform time
    num_samples = signals['noisy'].shape[-1]
    time_wave = np.arange(num_samples) / fs
    
    # Feature Time (if available)
    features = None
    feat_time = None
    if "feature_data" in data_dict:
        features = data_dict["feature_data"]["features"]
        num_frames = features.shape[1]
        feat_time = np.arange(num_frames) * hop / fs
        
    return {
        'fs': fs, 'hop': hop,
        'signals': signals, 'vads': vads,
        'labels': ordered_labels,
        'time_wave': time_wave,
        'features': features,
        'time_feat': feat_time,
        'prediction': data_dict.get("prediction_data"),
        'params': params,
        'refs': references
    }

# --- 4. LEFT PANEL: Aligned Plots ---
def create_left_plot(processed_data):
    """Creates the stacked figure with zero gaps and minimal ticks."""
    if not processed_data: return
    
    labels = processed_data['labels']
    sigs = processed_data['signals']
    vads = processed_data['vads']
    time_wave = processed_data['time_wave']
    
    feats = processed_data['features']
    feat_time = processed_data['time_feat']
    pred = processed_data['prediction']
    
    num_src = len(labels)
    # Rows: [Spec, Wave] per source, then [Feat1], [Spacer], [Feat2]
    total_rows = num_src * 2 + 3 
    
    # --- 1. Compute Unified Limits ---
    all_sigs_vals = [sigs[lbl] for lbl in labels]
    max_amp = 0
    if all_sigs_vals:
        max_amp = max(np.max(np.abs(s)) for s in all_sigs_vals if len(s) > 0)
    if max_amp == 0: max_amp = 1.0
    wave_ylim = [-max_amp * 1.1, max_amp * 1.1]

    # --- Setup Figure with 2 Columns ---
    height_ratios = []
    for _ in range(num_src):
        height_ratios.extend([2, 1])
    height_ratios.extend([4, 0.4, 4]) # Feat1, Spacer, Feat2
    
    fig_h = num_src * 1.5 + 4.5
    
    fig = plt.figure(figsize=(10, fig_h))
    # wspace=0.1 increased to allow space for secondary y-axis labels
    gs = fig.add_gridspec(total_rows, 2, width_ratios=[50, 1], 
                          height_ratios=height_ratios, hspace=0.0, wspace=0.1)
    
    # Create Main Column Axes
    axes = []
    for r in range(total_rows):
        sharex = axes[0] if r > 0 else None
        ax = fig.add_subplot(gs[r, 0], sharex=sharex)
        axes.append(ax)
    
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    label_colors = {lbl: colors[i % 10] for i, lbl in enumerate(labels)}
    
    spec_ims = []
    global_spec_max_db = -np.inf
    
    # --- Plot Signals ---
    for i, lbl in enumerate(labels):
        ax_spec = axes[i*2]
        ax_wave = axes[i*2 + 1]
        
        s = sigs[lbl]
        v = vads[lbl]
        c = label_colors[lbl]
        disp_lbl = lbl if not lbl.startswith("source") else f"S{lbl.split()[-1]}"
        
        # 1. Spectrogram
        Pxx, freqs, bins, im = ax_spec.specgram(s, NFFT=512, Fs=processed_data['fs'], 
                                                noverlap=256, cmap='inferno')
        spec_ims.append(im)
        
        if len(Pxx) > 0:
             pxx_safe = np.maximum(Pxx, 1e-10)
             curr_max = 10 * np.log10(np.max(pxx_safe))
             if curr_max > global_spec_max_db:
                 global_spec_max_db = curr_max
                 
        ax_spec.set_ylabel("Freq")
        
        # Moved Waveform label to Spectrogram top, removed spec label
        ax_spec.text(0.01, 0.8, f"{disp_lbl}", transform=ax_spec.transAxes, 
                     color=c, fontweight='bold')
        
        # 2. Waveform
        ax_wave.plot(time_wave, s, color=c, lw=0.8)
        
        # VAD Overlay
        if v is not None:
            if len(v) != len(time_wave):
                  v_t = torch.tensor(v).float().view(1,1,-1)
                  v_up = F.interpolate(v_t, size=len(time_wave), mode='nearest')
                  v = v_up.squeeze().numpy()
            ax_wave.fill_between(time_wave, -1e9, 1e9, where=(v > 0.5), 
                                 color=c, alpha=0.2, transform=ax_wave.get_xaxis_transform())
        
        ax_wave.set_ylabel("Amp")
        ax_wave.set_ylim(wave_ylim)

    # --- Unified Spectrogram Color Limits ---
    if global_spec_max_db == -np.inf: global_spec_max_db = 0
    spec_vmax = global_spec_max_db
    spec_vmin = spec_vmax - 80 
    
    for im in spec_ims:
        im.set_clim(spec_vmin, spec_vmax)
        
    # --- Colorbar 1: Spectrograms ---
    if spec_ims:
        cax_spec = fig.add_subplot(gs[0 : num_src*2, 1])
        fig.colorbar(spec_ims[0], cax=cax_spec, label='dB')

    # --- Plot Features ---
    idx_f1 = num_src * 2
    idx_spacer = idx_f1 + 1
    idx_f2 = idx_f1 + 2
    
    ax_f1 = axes[idx_f1]
    ax_spacer = axes[idx_spacer]
    ax_f2 = axes[idx_f2]
    
    # Hide spacer
    ax_spacer.set_visible(False)
    
    # Freq axis logic
    fs_val = processed_data['fs']
    f_min = fs_val / 1024
    f_max = 511 * fs_val / 1024
    
    vmin, vmax = None, None
    extent = [feat_time[0], feat_time[-1], f_min, f_max]
    feat_im = None
    
    if feats is not None:
        feats_np = feats.float().cpu().numpy()
        vmin = np.percentile(feats_np, 1)
        vmax = np.percentile(feats_np, 99)
        
        feat_im = ax_f1.imshow(feats_np[:511, :], aspect='auto', origin='lower', extent=extent,
                               cmap='inferno', vmin=vmin, vmax=vmax)
        ax_f1.set_ylabel("wGMSC")
        # ax_f1.set_title("Normal wGMSC")
        
        ax_f2.imshow(feats_np[511:, :], aspect='auto', origin='lower', extent=extent,
                     cmap='inferno', vmin=vmin, vmax=vmax)
        ax_f2.set_ylabel("wGMSC (rev)")
        ax_f2.set_xlabel("Time [s]")
        
        # Calculate GT
        num_frames = feats.shape[1]
        gt_count = np.zeros(num_frames)
        for i, lbl in enumerate(labels):
            if lbl in ['noisy', 'noise']: continue
            v = vads.get(lbl)
            if v is not None:
                v_t = torch.tensor(v).float().view(1,1,-1)
                v_res = F.interpolate(v_t, size=num_frames, mode='nearest')
                gt_count += v_res.squeeze().numpy()
        
        p_np = None
        if pred is not None:
            p_np = pred.float().cpu().numpy()
            
            acc = np.mean(np.round(p_np) == gt_count)
            mae = np.mean(np.abs(p_np - gt_count))
            mse = np.mean((p_np - gt_count)**2)
            
            ax_f2.set_title(f"Acc: {acc:.1%} | MAE: {mae:.2f} | MSE: {mse:.2f}", fontsize=10, y=0.97)
            
            # --- New: Add overlay to upper plot as well ---
            # Using same metrics or simplified title for upper plot? 
            # User said "legend y labels and just everything like in the lower plot"
            # But the lower plot has specific title. Let's keep the upper plot title as is ("Normal wGMSC") unless requested.
            
        mx_c = max(np.max(gt_count), np.max(p_np) if p_np is not None else 0, 3)

        # Helper function to add overlay
        def add_overlay(target_ax):
            ax_t = target_ax.twinx()
            ax_t.plot(feat_time, gt_count, color='cyan', linestyle='-', lw=2, label='GT')
            if p_np is not None:
                ax_t.plot(feat_time, p_np, 'w--', lw=2, label='Pred')
            
            ax_t.set_ylabel("Count")
            ax_t.grid(False, axis='y')
            ax_t.grid(False, axis='x') 
            ax_t.legend(loc='upper right', frameon=True, facecolor='black', framealpha=0.6, labelcolor='white')
            ax_t.set_ylim(-0.5, mx_c + 0.5)

        # Apply to both feature plots
        add_overlay(ax_f1)
        add_overlay(ax_f2)

        # --- Colorbar 2: Features ---
        if feat_im:
            cax_feat = fig.add_subplot(gs[idx_f1 : idx_f2 + 1, 1])
            fig.colorbar(feat_im, cax=cax_feat)

    # --- Clean up ticks & grids ---
    # User Requirement: 
    # 1. remove all gaps (done via hspace=0 and spacer)
    # 2. remove x labels and x ticks except from the lowest feature plot (ax_f2)
    # 3. remove all horizontal grid lines
    
    for i, ax in enumerate(axes):
        # Skip spacer
        if i == idx_spacer: continue
        
        # Remove horizontal grid
        ax.yaxis.grid(False) 
        # Ensure vertical grid is roughly present if style allows (darkgrid usually has it)
        # We can force it:
        ax.xaxis.grid(True)

        # Remove x ticks/labels for all except last
        if i != idx_f2:
            ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
            ax.set_xlabel("")
    
    # Final x-limit
    axes[0].set_xlim(time_wave[0], time_wave[-1])
    plt.show()

# --- 5. RIGHT PANEL: Info & Audio ---
def create_right_panel(processed_data):
    """Spatial plots (3D & 2D), Table, Audio."""
    if not processed_data: return
    
    params = processed_data['params']
    room_dims = params.get('room_dims', [0,0,0])
    mic_pos = np.array(params.get('mic_pos', []))
    srcs = np.array(params.get('source_positions', []))
    
    # Fix Mic Shape to (M, 3) for plotting logic
    mics = None
    if mic_pos.ndim == 2:
        if mic_pos.shape[0] == 3 and mic_pos.shape[1] != 3: 
            mics = mic_pos.T 
        else:
            mics = mic_pos
    else:
        mics = mic_pos
        
    center = np.mean(mics, axis=0) if len(mics) > 0 else [0,0,0]
    L, W, H = room_dims
    
    # --- 1. Audio & Table (Top Row) ---
    out_audio = widgets.Output(layout={'width': '40%'})
    out_table = widgets.Output(layout={'width': '60%'})
    
    with out_audio:
        # fs = processed_data['fs']
        # Use Helper for compact audio display
        def show_audio(label, sig):
            display(HTML(f"<div style='margin-bottom: 1px; margin-top: 5px; font-weight: bold; font-size: 10pt;'>{label}</div>"))
            display(Audio(sig, rate=fs))

        fs = processed_data['fs']
        
        # Audio Section logic
        mix = processed_data['signals']['noisy']
        if mix.ndim > 1: mix = mix[0] 
        # First one gets less top margin
        display(HTML(f"<div style='margin-bottom: 1px; margin-top: 0px; font-weight: bold; font-size: 10pt;'>🔊 Mixture</div>"))
        display(Audio(mix, rate=fs))
        
        refs = processed_data['refs']
        for k in processed_data['labels']:
            if k == 'noisy': continue
            if k in refs:
                s = refs[k]
                if isinstance(s, torch.Tensor): s = s.float().cpu().numpy()
                if s.ndim > 1: s = s[0]
                show_audio(f"🔊 {k}", s)

    with out_table:
        # Metadata Table logic
        table_data = []
        include_keys = ['scenario_id', 'generator_id', 'mic_array', 'num_sources', 'signal_length', 'snr', 'rt60', 'room_dims']
        for key in include_keys:
            if key in params:
                value = params[key]
                if isinstance(value, float): display_value = f"{value:.2f}"
                elif isinstance(value, list) and len(value) > 0 and isinstance(value[0], float): display_value = f"[{', '.join([f'{v:.2f}' for v in value])}]"
                else: display_value = value
                display_key = key
                if key == 'mic_array': display_key = 'Mic Array Geometry/Name'
                table_data.append({"Parameter": display_key, "Value": display_value})

        radius = 0
        if len(mics) > 0:
            radii = np.linalg.norm(mics - center, axis=1)
            radius = np.max(radii)

        table_data.append({"Parameter": "Mic Array Position (Center)", "Value": f"[{', '.join([f'{c:.2f}' for c in center])}]"})
        table_data.append({"Parameter": "Mic Array Radius (Approx)", "Value": f"{radius:.3f}"})

        if len(srcs) > 0:
            formatted_srcs = []
            for i, pos in enumerate(srcs):
                pos_str = f"[{', '.join([f'{p:.2f}' for p in pos])}]"
                doa_str = ""
                if len(center) == 3 and len(pos) == 3:
                    vec = pos - center
                    az = np.degrees(np.arctan2(vec[1], vec[0]))
                    xy_dist = np.linalg.norm(vec[:2])
                    el = np.degrees(np.arctan2(vec[2], xy_dist))
                    doa_str = f" (Az: {az:.1f}°, El: {el:.1f}°)"
                formatted_srcs.append(f"<b>Src {i+1}:</b> {pos_str}{doa_str}")
            table_data.append({"Parameter": "Source Positions", "Value": "<br>".join(formatted_srcs)})

        if 'sirs' in params:
            sirs = params['sirs']
            if isinstance(sirs, list):
                 display_sirs = f"[{', '.join([f'{s:.2f}' for s in sirs])}]"
                 table_data.append({"Parameter": "Source-to-Interference Ratios (SIRs)", "Value": display_sirs})

        if 'noise_file_path' in params:
            noise_path_str = params['noise_file_path']
            if noise_path_str:
                 noise_path = Path(noise_path_str)
                 noise_info = f"{noise_path.parent.name}/{noise_path.name}"
                 table_data.append({"Parameter": "Noise File", "Value": noise_info})

        df = pd.DataFrame(table_data)
        display(HTML(df.to_html(index=False, escape=False, classes='table table-striped')))
    
    display(widgets.HBox([out_audio, out_table]))
    
    # --- 2. Room Plots (Bottom Row) ---
    out_3d = widgets.Output(layout={'width': '100%'})
    
    with out_3d:
        # Plotly 3D Room
        # Wireframe logic
        x_l, y_l, z_l = [], [], []
        # Bottom
        x_l += [0, L, L, 0, 0]; y_l += [0, 0, W, W, 0]; z_l += [0, 0, 0, 0, 0]
        x_l.append(None); y_l.append(None); z_l.append(None)
        # Top
        x_l += [0, L, L, 0, 0]; y_l += [0, 0, W, W, 0]; z_l += [H, H, H, H, H]
        x_l.append(None); y_l.append(None); z_l.append(None)
        # Verticals
        for x, y in [(0,0), (L,0), (L,W), (0,W)]:
            x_l += [x, x, None]
            y_l += [y, y, None]
            z_l += [0, H, None]
            
        fig3d = go.Figure()
        fig3d.add_trace(go.Scatter3d(
            x=x_l, y=y_l, z=z_l, mode='lines', line=dict(color='black', width=3), name='Room', hoverinfo='skip'
        ))
        
        if len(mics) > 0:
            fig3d.add_trace(go.Scatter3d(
                x=mics[:,0], y=mics[:,1], z=mics[:,2], mode='markers', 
                marker=dict(size=4, color='blue'), name='Mics'
            ))
            
        fig3d.add_trace(go.Scatter3d(
            x=[center[0]], y=[center[1]], z=[center[2]], mode='markers',
            marker=dict(size=4, color='red'), name='Center'
        ))
        
        if len(srcs) > 0:
            fig3d.add_trace(go.Scatter3d(
                x=srcs[:,0], y=srcs[:,1], z=srcs[:,2], mode='markers+text',
                marker=dict(size=6, color='green', symbol='diamond'),
                text=[f"S{i+1}" for i in range(len(srcs))], name='Srcs'
            ))
        
        # Calculate height dynamically based on left panel height
        num_src = len(processed_data['labels'])
        # Left panel formula in inches: num_src * 1.5 + 4.5
        # Convert to pixels (approx 90 dpi) and subtract top panel height estimate (~350px)
        left_h_inches = num_src * 1.5 + 4.5
        target_h = int(left_h_inches * 90 - 350)
        if target_h < 500: target_h = 500
            
        fig3d.update_layout(
            scene=dict(
                xaxis=dict(range=[0, L], title='X'),
                yaxis=dict(range=[0, W], title='Y'),
                zaxis=dict(range=[0, H], title='Z'),
                aspectmode='data'
            ),
            margin=dict(l=20, r=20, b=20, t=60), # Increased top margin
            height=target_h,
            title=dict(text="Interactive 3D View", y=0.95),
            legend=dict(yanchor="top", y=0.9, xanchor="right", x=0.9) # Explicit legend
        )
        display(go.FigureWidget(fig3d))
        
    display(out_3d)

# --- 6. DASHBOARD APP ---
w_scenario_id = widgets.IntText(value=0, description='Scenario ID:')
w_btn_load = widgets.Button(description="Load Scenario", button_style='primary')
w_out_left = widgets.Output(layout={'width': '50%', 'border': '1px solid #ccc'})
w_out_right = widgets.Output(layout={'width': '50%', 'border': '1px solid #ccc', 'padding': '10px'})
w_status = widgets.Label(value="Ready.")

def on_load_click(b):
    w_status.value = "Loading..."
    scen_id = w_scenario_id.value
    
    data_dict, msg = load_data(scen_id)
    
    if data_dict is None:
        w_status.value = f"Error: {msg}"
        return
        
    w_status.value = "Processing..."
    try:
        proc_data = preprocess_data(data_dict)
        
        with w_out_left:
            clear_output(wait=True)
            create_left_plot(proc_data)
            
        with w_out_right:
            clear_output(wait=True)
            create_right_panel(proc_data)
            
        w_status.value = f"Loaded Scenario {scen_id}"
    except Exception as e:
        w_status.value = f"Error processing: {str(e)}"
        import traceback
        traceback.print_exc()

w_btn_load.on_click(on_load_click)

header = widgets.HBox([w_scenario_id, w_btn_load, w_status])
main_view = widgets.HBox([w_out_left, w_out_right])
dashboard = widgets.VBox([header, main_view])

display(dashboard)